# demo_c1

> Phase 20 (`specs/multiagent/phase20/`, experiment C1): squads with coupled work and bridges at 64 agents. Quick (the fake client and synthetic rows), the pilot, the main run and Gate C1 (live, Claude Haiku 5.5), and part b and the 3B bridge check (from the Colab zip), under `C1_PILOT_PLAN` and `C1_PLAN`.

Artifacts (gitignored): `results/multiagent/colony/{c1quick, c1pilot, c1main1, c1main2, c1, c1b}/` (`runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `c1.png`, `summary.json`; `c1/reused.csv`). The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), under the cap of $250 per calendar month (P20-9). Rows carry Phase 18's, the bridge and Phase 19's columns (`bridge_mech.runC1Cell`).

- **Quick** (normal tests): the three pilot cells for 3 turns with the fake client (C1's columns), then the analysis, report and `c1.png` on `syntheticRows20` and `syntheticPartB`, in `c1quick/`.
- **Pilot** (`--flags llm`, `EXPLRL_LLM_LIVE=1`, `EXPLRL_C1_STAGE=pilot`; about $2): `coupled`, `bridge1` and `bridgeGuard` at seed 1890. Needs `C1_PILOT_PLAN` frozen (after the user approved the estimate, P20-8). Then `pilotCheck20`: stop and ask on any failed check, and report the projection and wait.
- **Main** (`--cell-timeout 0 --flags llm`, `EXPLRL_LLM_LIVE=1`, `EXPLRL_C1_STAGE=main1`, then `main2`, then `main`; about $25-31): the 38 new boards in two seed batches, then the full grid from the cache ($0) with the 16 reused boards, `c1Summaries`, the report and `c1.png`. Needs `C1_PLAN` frozen on the pilot plan's hash, B3's forecast with its SHA-256 in `C1_PLAN` (P20-13) and the user's approval of the projection recorded in `C1_PLAN['projection']['approved']` (P20-8).
- **Part b** (`--flags train`, $0): unzip `explRL_c1b.zip` into `results/multiagent/rl/colab/`; `partBSummaries` on the squads-of-4 rows and the reused one-board rows (`reusedPartB`), and the bridge check's relays and longest prompts, into `c1b/report.md`.

In [ ]:
import copy
import hashlib
import json
import os
import re
import shutil
import subprocess
import time

from explRL.bench_grid import readCSV, writeCSV
from explRL.bridge_grid import c1Grid, c1PilotGrid, c1Range, reusedCells20, reusedPartB
from explRL.bridge_mech import reusedRows20, runC1Cell
from explRL.bridge_plan import C1_PILOT_PLAN, C1_PLAN, pilotCheck20
from explRL.bridge_report import c1Summaries, partBSummaries
from explRL.bridge_write import figC1, syntheticPartB, syntheticRows20, writeC1
from explRL.coop_grid import coopHash
from explRL.liar_trace import cacheOnlyClient
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.train import demo_artifact_dir

COLONY = 'multiagent/colony'
WORKERS, LLM_WORKERS = min(8, os.cpu_count() or 1), 32
LLM = demo_artifact_dir(None, 'multiagent/llm')
COLAB = demo_artifact_dir(None, 'multiagent/rl') / 'colab'
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
STAGE = os.environ.get('EXPLRL_C1_STAGE', 'pilot')        # what a live run may do: pilot, main1, main2 (seed batches) or main
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=float(C1_PLAN['cap']), ledger=str(LLM / 'spend.jsonl'))
FORECAST = LLM.parents[2] / C1_PLAN['forecast']['file']
BATCHES20 = {'main1': C1_PLAN['seeds']['eight'][:4], 'main2': C1_PLAN['seeds']['eight'][4:]}   # every batch runs; not analysed alone


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def runC1(mode, cells, spec, plan, workers=None):
    "C1 cells through runWorkload (runC1Cell): runs.csv, turns.jsonl, timing.jsonl and metrics.json; live runs keep *_live copies."
    live, out = spec.get('kind') == 'live', demo_artifact_dir(mode, COLONY)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload([(cells[0]['study'], cells)], spec, workers or (LLM_WORKERS if live else WORKERS), str(logs), run=runC1Cell)
    wall, rows = time.time() - t0, res['rows']
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    meta = dict(mode=mode, planHash=coopHash(plan), commit=gitCommit(), model=HAIKU, cap=float(spec.get('cap', 0.0)), wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)), calls=int(sum(r['calls'] for r in rows)),
                spent=Budget(spec['cap'], spec['ledger']).spent() if spec.get('ledger') else 0.0)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'], timing=timing),
                                                 indent=2, sort_keys=True, default=str) + '\n')
    if live:
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent this month, "
          f"{wall:.0f} s; stopped: {res['stopped']}")
    return rows, meta


def analyse(mode, rows, meta, plan=C1_PLAN, nboot=None, partB=None):
    "c1Summaries, report.md, c1.png and summary.json in the mode's folder."
    out = demo_artifact_dir(mode, COLONY)
    S = c1Summaries(rows, plan, nboot=nboot)
    writeC1(out / 'report.md', S, meta, plan, partB=partB)
    figC1(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, summary=S, partB=partB), indent=1, sort_keys=True, default=str) + '\n')
    return S


def mainReady():
    """AC-8 and P20-8, P20-13 before the main run: C1_PLAN frozen on the frozen pilot plan's hash, B3's forecast committed with
    the SHA-256 recorded in C1_PLAN, and the user's approval of the pilot's projection recorded (C1_PLAN['projection']['approved'])."""
    f = C1_PLAN['forecast']
    return [m for m, bad in (
        ('C1_PILOT_PLAN is not frozen', not C1_PILOT_PLAN['frozen']), ('C1_PLAN is not frozen', not C1_PLAN['frozen']),
        ("C1_PLAN['pilotHash'] is not the frozen pilot plan's hash", C1_PLAN['pilotHash'] != coopHash(C1_PILOT_PLAN)),
        ("B3's forecast is missing or its SHA-256 differs from C1_PLAN's", not (f['sha256'] and FORECAST.exists() and
                                                                            hashlib.sha256(FORECAST.read_bytes()).hexdigest() == f['sha256'])),
        ("the user's approval of the pilot's projection is not recorded", not (C1_PLAN['projection'] or {}).get('approved'))) if bad]

## Quick: the fake client and synthetic rows

In [ ]:
t0, q = time.time(), demo_artifact_dir('c1quick', COLONY)
# the pipeline with the fake client: the three pilot cells (64 agents, coupled and bridged) for 3 turns, with C1's columns
cells = [dict(c, cfg=dict(copy.deepcopy(c['cfg']), turns=3)) for c in c1PilotGrid()]
frows, _ = runC1('c1quick', cells, dict(kind='fake'), C1_PILOT_PLAN)
assert len(frows) == 3 and all(k in r for r in frows for k in ('honestPooled', 'mirrorPosts', 'bridgeRelays', 'crossContamination', 'agentTurns'))
assert {r['level'] for r in frows} == {f'{a}|independent' for a in C1_PILOT_PLAN['cells']['pilot']}
# the analysis on synthetic rows with known effects (every planned cell; part b's rows too)
rows, prows = syntheticRows20(), syntheticPartB()
writeCSV(rows, q / 'synthetic.csv')
B = partBSummaries(prows)
S = analyse('c1quick', rows, dict(mode='quick', planHash=coopHash(C1_PLAN), commit=gitCommit()), nboot=200, partB=B)
assert S['gate']['passes'] and not S['inert']['fires'] and S['hyp']['H20_3']['supported'] and B['H20_6']['p'] == 1 / 256
for f in ('runs.csv', 'turns.jsonl', 'metrics.json', 'synthetic.csv', 'report.md', 'c1.png', 'summary.json'):
    assert (q / f).stat().st_size > 0, f
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
assert time.time() - t0 < 300
print(open(q / 'report.md').read()[:2500])

## Pilot (`--flags llm`, live, about $2)

Seed 1890, one 64-agent board each of `coupled`, `bridge1` and `bridgeGuard` (not analysed). `pilotCheck20`: invalid replies under 5%, mirror posts, relays on each bridge board, bridge input tokens per call at most 1,560; the projection of the main run beside the $35 ceiling; the verdict is always 'ask'.

In [ ]:
#| llm
if not LIVE or STAGE != 'pilot':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_C1_STAGE=pilot')
else:
    assert C1_PILOT_PLAN['frozen'], 'report the estimate, get the approval, then freeze and push C1_PILOT_PLAN before the pilot (AC-8, P20-8)'
    print('pilot estimate (low, high):', c1Range()['pilot'])
    prows, pmeta = runC1('c1pilot', c1PilotGrid(), LIVE_SPEC, C1_PILOT_PLAN)
    P = pilotCheck20(prows)
    demo_artifact_dir('c1pilot', COLONY).joinpath('report.md').write_text(f"# C1 pilot (not analysed)\n\n{json.dumps(P, indent=1, default=str)}\n")
    print(f"checks {'pass' if P['ok'] else 'FAIL: stop and ask'} {P['reasons']}; cost per call by arm {P['perCall']}; projected main run "
          f"${P['projected']:.2f} against the ${P['ceiling']:.0f} ceiling: report to the user and wait for approval ({P['verdict']})")

## Main run and Gate C1 (`--cell-timeout 0 --flags llm`, live, about $25-31; then $0)

`main1` and `main2` play the 38 new boards by seed batch (1800-1803, 1804-1807); between them the only stops are 5% invalid replies and the cap. `main` replays the full grid from the cache, replays the 16 reference boards with C1's columns (`reusedRows20`, earlier columns unchanged) and writes the frozen analysis.

In [ ]:
#| llm
def reused20():
    "The 16 reference boards replayed from the cache-only client with C1's columns, earlier columns checked (P20-6): reused.csv."
    prior = readCSV(demo_artifact_dir('stage4', COLONY) / 'runs.csv') + readCSV(demo_artifact_dir('stage64', COLONY) / 'runs.csv')
    rows = reusedRows20(lambda cfg: cacheOnlyClient(cfg, LIVE_SPEC['cache']), prior, reusedCells20())
    writeCSV(rows, demo_artifact_dir('c1', COLONY) / 'reused.csv')
    return rows


def invalidShare(rows):
    return sum(r['invalid'] for r in rows) / (sum(r['calls'] for r in rows) or 1)


if not LIVE or STAGE not in ('main1', 'main2', 'main'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_C1_STAGE=main1, main2 or main')
else:
    bad = mainReady()
    assert not bad, f'the main run waits (AC-8, P20-8, P20-13): {bad}'
    if STAGE in BATCHES20:
        b1 = demo_artifact_dir('c1main1', COLONY) / 'runs.csv'
        assert STAGE == 'main1' or (b1.exists() and invalidShare(readCSV(b1)) < C1_PLAN['pilot']['maxInvalid']), \
            'batch 2 waits for batch 1 with invalid replies under 5% (stop and ask)'
        brows, _ = runC1(f'c1{STAGE}', [c for c in c1Grid() if c['seed'] in BATCHES20[STAGE]], LIVE_SPEC, C1_PLAN)
        print(f'{STAGE} (not analysed): {len(brows)} boards, invalid {invalidShare(brows):.3f}')
    else:                                                  # the full grid: every reply cached by the batches ($0; rows keep the live cost)
        spent = Budget(LIVE_SPEC['cap'], LIVE_SPEC['ledger']).spent()
        nrows, meta = runC1('c1', c1Grid(), LIVE_SPEC, C1_PLAN)
        assert Budget(LIVE_SPEC['cap'], LIVE_SPEC['ledger']).spent() == spent, 'the full grid must replay from the cache at $0'
        S = analyse('c1', nrows + reused20(), meta)
        print(open(demo_artifact_dir('c1', COLONY) / 'report.md').read()[:6000])

## Part b and the 3B bridge check (`--flags train`, $0)

H20-6 (descriptive): per `r17mix` run, the welfare gain over the warm start in squads of 4 minus on one board, at 32 agents. P20-11: relays per bridge turn, mirror posts and invalid replies of the warm start in A1's world, and its longest prompt per role.

In [ ]:
#| train
p, pc = COLAB / 'eval_c1b' / 'runs.csv', COLAB / 'check_c1'
if not p.exists():
    print('skipped: unzip explRL_c1b.zip into results/multiagent/rl/colab/ first')
else:
    new, old = readCSV(p), reusedPartB(root=COLAB)               # old: the one-board rows, configuration checked
    assert len(new) == 180 and len(old) == 180, (len(new), len(old))
    B = partBSummaries(new + old)
    out = demo_artifact_dir('c1b', COLONY)
    writeC1(out / 'report.md', None, dict(mode='partb', planHash=coopHash(C1_PILOT_PLAN), commit=gitCommit()), partB=B)
    check = readCSV(pc / 'runs.csv')
    assert len(check) == 10, len(check)
    for lv in sorted({r['level'] for r in check}):
        rs = [r for r in check if r['level'] == lv]
        print(lv, {k: round(sum(float(r[k]) for r in rs) / len(rs), 3) for k in ('relaysPerBridgeTurn', 'mirrorPosts', 'bridgeRelays', 'invalid')})
    print('longest prompt per role:', json.loads((pc / 'prompts.json').read_text()))
    print(open(out / 'report.md').read()[:3000])